[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tony921023/liver-fibrosis-test/blob/main/colab_setup.ipynb)

# Liver Fibrosis — Colab 訓練

在 Colab 跑 `train.py`。程式碼本身不用改:`get_device()` 會自動選 CUDA,
`DATA_DIR` 用環境變數覆蓋。

## ⚠️ 坑之一:別把 DATA_DIR 指到 Google Drive

`config.DEDUP=True` 會對每個檔案算內容 hash(這是本專案最重要的一步,見 `dataset.py` 的
LEAKAGE 警告)。6323 個檔案在**本地磁碟**約 5 秒,但如果 `DATA_DIR` 直接指到
**掛載的 Google Drive**,6323 次檔案開啟走 FUSE 會慢到幾分鐘起跳,而且每次重跑都要再等一次。

→ **一定要把資料解壓/複製到 `/content`,不要直接讀 Drive。**

## ⚠️ 坑之二:相對路徑會被 `%cd` 咬到

`%cd` 之後,所有相對路徑的基準都跟著變。所以本 notebook 一律用**絕對路徑**:
程式碼在 `/content/test`、資料在 `/content/dl`,兩者互不干擾。

也因此**不需要手動把資料 `mv` 進 repo 的 `data/`** —— 第 4 節會自動找出資料在哪,
再用環境變數告訴 `config.py`。

## 執行順序
由上而下跑一次即可。第 3 節(取得資料)有三種來源,擇一。

## 1. 確認 GPU

沒有 GPU 的話:「執行階段 → 變更執行階段類型 → T4 GPU」。
CPU 也跑得動(資料量小),只是會慢很多。

In [ ]:
!nvidia-smi || echo '沒有 GPU,會退回 CPU'
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 2. 取得程式碼

In [ ]:
import pathlib

REPO = '/content/test'

# 冪等:第一次 clone,之後重跑就強制同步到最新的 main。
# 若直接 git clone 到已存在的目錄會失敗,但 %cd 仍會成功 → 整份 notebook 靜悄悄跑舊碼。
if pathlib.Path(REPO, '.git').exists():
    print('repo 已存在,強制同步到 origin/main(會丟棄本地改動,例如 sed 過的 config.py)')
    !git -C {REPO} fetch --quiet origin
    !git -C {REPO} reset --hard --quiet origin/main
else:
    !git clone --quiet https://github.com/tony921023/liver-fibrosis-test.git {REPO}

%cd {REPO}
!git log -1 --oneline

# Colab 已預裝 torch/torchvision/sklearn/matplotlib/pandas/numpy,
# 這行實際上只會補裝 kaggle。requirements.txt 用寬鬆版本,不會把 Colab 的 torch 降級。
!pip install -q -r requirements.txt

## 3. 取得資料(三選一)

`data/` 在 `.gitignore` 裡,clone 下來是空的,要另外抓。
**三種方式的共同目標:讓 F0~F4 五個資料夾出現在 `/content` 底下的某處。**

### 3a. 從 Kaggle 下載(推薦)

資料集:[vibhingupta028/liver-histopathology-fibrosis-ultrasound-images](https://www.kaggle.com/datasets/vibhingupta028/liver-histopathology-fibrosis-ultrasound-images)(205 MB,CC-BY-SA-4.0)

**憑證優先走 Colab Secrets**(左邊欄的 🔑 圖示),設定一次就好:

1. 點左邊欄 **🔑** → **新增密鑰**
2. 建兩個:`KAGGLE_USERNAME` 和 `KAGGLE_KEY`,值取自你的 `kaggle.json`
3. 兩個都打開「**筆記本存取權**」

讀不到 Secrets 時會退回 `files.upload()` 上傳 `kaggle.json`。但那個 widget 綁定
當下的瀏覽器 session,**runtime 一斷線重連就會失效**(出現 "Upload widget is only
available when the cell has been executed in the current browser session"),
只能重跑該格並立刻選檔。Secrets 沒有這個問題。

---

**注意 `-p` 用絕對路徑 `/content/dl`。** 若寫成相對路徑 `-p data`,下載位置會跟著當下的
cwd 跑(`%cd` 過之後就對不上了),第 4 節反而找不到。

解壓後的結構是 `/content/dl/Dataset/Dataset/F0..F4`(zip 裡本來就多包一層),
**不用手動 `mv` 搬平** —— 第 4 節會自動搜尋含 F0~F4 的那一層。

In [ ]:
import os, json, pathlib

DL = pathlib.Path('/content/dl')
SLUG = 'vibhingupta028/liver-histopathology-fibrosis-ultrasound-images'

# --- 憑證:優先 Colab Secrets,退回上傳 kaggle.json ---
# kaggle CLI 直接讀 KAGGLE_USERNAME / KAGGLE_KEY 這兩個環境變數,不一定要有 kaggle.json
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('✅ 憑證來自 Colab Secrets')
except Exception as e:
    print(f'讀不到 Secrets({type(e).__name__}),改用上傳 kaggle.json')
    print('⚠️ widget 若顯示 "Please rerun this cell to enable",'
          '代表 runtime 重連過 → 重跑本格並立刻選檔')
    from google.colab import files
    files.upload()
    creds = json.loads(pathlib.Path('kaggle.json').read_text())
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']
    print('✅ 憑證來自 kaggle.json')

# --- 下載(已下載過就跳過,重跑本格不會又抓一次 205 MB)---
already = DL.exists() and any(DL.rglob('F0'))
if already:
    print('✅ /content/dl 已有資料,跳過下載')
else:
    # "outdated kaggle version" 的警告可以忽略,不影響下載
    !kaggle datasets download -d {SLUG} -p /content/dl --unzip

!find /content/dl -maxdepth 3 -type d | head -20

### 3b. 從 Google Drive 複製

注意是**複製到 `/content`**,不是直接拿 Drive 路徑當 `DATA_DIR`(見開頭的坑)。
如果 Drive 上放的是 zip,解壓到 `/content` 會比複製幾千個小檔快得多。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 情況一:Drive 上是 zip(快)
!mkdir -p /content/dl && unzip -q '/content/drive/MyDrive/<路徑>/dataset.zip' -d /content/dl

# 情況二:Drive 上是解開的資料夾(慢,幾千個小檔)
# !cp -r '/content/drive/MyDrive/<路徑>/Dataset' /content/dl/

!find /content/dl -maxdepth 3 -type d | head -20

### 3c. 直接上傳 zip

In [ ]:
from google.colab import files
up = files.upload()     # 選你的 dataset zip

import pathlib
zname = next(iter(up))
!mkdir -p /content/dl && unzip -q "{zname}" -d /content/dl
!find /content/dl -maxdepth 3 -type d | head -20

## 4. 自動找出 F0~F4 的所在目錄,設成 DATA_DIR

不同來源解壓出來的巢狀層數常常不一樣,所以用搜尋的,不要手寫死路徑。

`os.environ` 的改動會被 `!python train.py` 這種 shell 子行程繼承,所以設一次就好,
不用改 `config.py`。

In [ ]:
import os, pathlib

# 先排除 drive/,不要遞迴進去 —— 掛載的 Drive 走 FUSE,rglob 爬起來會非常慢
bases = [p for p in pathlib.Path('/content').iterdir()
         if p.is_dir() and p.name not in {'drive', 'sample_data'}]

root = None
for base in bases:
    for p in base.rglob('F0'):
        if p.is_dir() and all((p.parent / f'F{i}').is_dir() for i in range(5)):
            root = p.parent
            break
    if root:
        break

assert root is not None, f'在 {[str(b) for b in bases]} 裡找不到同時含 F0~F4 的目錄,請檢查第 3 節'
os.environ['DATA_DIR'] = str(root)   # 會被 !python train.py 子行程繼承
print('DATA_DIR =', root)

total = 0
for i in range(5):
    n = sum(1 for _ in (root / f'F{i}').iterdir())
    total += n
    print(f'  F{i}: {n} files')
print(f'  total: {total}   (期望 6323)')

## 5. 資料健檢:確認 Kaggle 這份跟本機那份是同一個東西

本機那份的事實(見 `CLAUDE.md`):

| 項目 | 期望值 |
|---|---|
| 檔案數 | 6323 |
| **不重複影像數** | **1536** |
| 跨類別重複的 hash | 0(標籤沒有矛盾) |

**如果 unique 數對不上,先搞清楚為什麼再訓練** —— 表示 Kaggle 版本跟本機這份不同,
後面所有數字都不能跟舊 run 比較。

In [ ]:
import hashlib, collections, time, pathlib, os

root = pathlib.Path(os.environ['DATA_DIR'])
t = time.time()
hash_to_classes = collections.defaultdict(set)
n = 0
for i in range(5):
    for f in sorted((root / f'F{i}').iterdir()):
        n += 1
        hash_to_classes[hashlib.md5(f.read_bytes()).hexdigest()].add(i)

cross = sum(1 for v in hash_to_classes.values() if len(v) > 1)
print(f'耗時 {time.time() - t:.1f}s   (本地磁碟應該 <15s;若破分鐘,代表你在讀 Drive)')
print(f'檔案數        : {n}              (期望 6323)')
print(f'不重複影像數  : {len(hash_to_classes)}   (期望 1536)')
print(f'跨類別重複    : {cross}              (期望 0)')

## 6. 訓練

Colab 通常有 4~8 個 vCPU,`NUM_WORKERS` 可以比 Mac 上調大。

resnet18 + 1074 張訓練圖 + 30 epochs,在 T4 上大約幾分鐘。

### 首輪 baseline 實測(resnet18,尚未加正則化)

| 指標 | 值 |
|---|---|
| macro AUROC | 0.9398 |
| balanced accuracy | **0.7506** ← 真正該看的數字 |
| QWK | 0.8195 |

macro AUROC 在 5 分類 one-vs-rest 下偏寬鬆(F0 太好分,把平均拉高)。
而且 **patient-level leakage 還在**,所以 0.94 仍偏樂觀。

### 判讀

- **看到 0.99** → `DEDUP` 被關掉了,模型在背圖,不是在判讀
- **train_loss 一路降、val_loss 卡住** → 過擬合。已加 dropout / weight decay / mixup 對抗
- ⚠️ **mixup 開啟時 `train_loss` 會偏高**(target 是混合的),不能直接跟 `val_loss` 比大小,
  要看 `val_loss` 自己的走勢有沒有回升

In [ ]:
%cd /content/test

# Colab 通常有 4~8 個 vCPU,可以比 Mac 上調大
!sed -i 's/^NUM_WORKERS = .*/NUM_WORKERS = 4                # Colab: 比 Mac 調大/' config.py
!grep -E '^(DEDUP|BACKBONE|EPOCHS|NUM_WORKERS|TTA|AUG_STRENGTH) ' config.py

!python train.py

## 6b. 兩個任務一起跑

`TASK` 和 `RESULTS_DIR` 都吃環境變數,所以同一份 code 可以跑兩種預測目標、結果分開存:

| TASK | 問的問題 | 主要指標 |
|---|---|---|
| `multiclass` | METAVIR 五分期 F0~F4 | macro AUROC、QWK |
| `binary_geF2` | **有沒有顯著纖維化(≥F2)** | **sensitivity / specificity** |

`binary_geF2` 才是臨床上真正在問的問題。首輪 baseline 把 5 分類結果直接摺成二元,
大約是 sensitivity 0.855 / specificity 0.839 —— 這次是真的用二元目標重新訓練。

In [ ]:
%cd /content/test

# 第 7 節的 sed 會把 BACKBONE 留在迴圈最後一個值(convnext_tiny),先還原
!git checkout -- config.py
!sed -i 's/^NUM_WORKERS = .*/NUM_WORKERS = 4/' config.py
!grep -E '^(BACKBONE|TASK|DROPOUT|WEIGHT_DECAY|MIXUP_ALPHA) ' config.py

import json, os, pathlib, subprocess

runs = {
    'multiclass':  'outputs_multiclass',
    'binary_geF2': 'outputs_binary',
}

reports = {}
for task, outdir in runs.items():
    print(f'\n{"=" * 60}\n  TASK = {task}\n{"=" * 60}')
    env = {**os.environ, 'TASK': task, 'RESULTS_DIR': outdir}
    subprocess.run(['python', 'train.py'], env=env, check=True)

    r = json.loads((pathlib.Path(outdir) / 'test_report.json').read_text())

    # 防呆:確認跑的真的是這個 task、而且用的是有正則化的新版 train.py。
    # (舊版 config.py 把 TASK/RESULTS_DIR 寫死,環境變數不會生效,會靜悄悄給出錯的報告)
    assert r['task'] == task, f"報告寫 task={r['task']},但要求的是 {task} → 你在跑舊碼,回第 2 節重跑"
    assert 'mixup_alpha' in r, "報告缺 mixup_alpha 欄位 → 你在跑舊碼,回第 2 節重跑"

    reports[task] = r

print(f'\n{"=" * 60}\n  總結\n{"=" * 60}')
for task, r in reports.items():
    print(f'\n[{task}]  backbone={r["backbone"]}  best_epoch={r["best_epoch"]}  '
          f'dropout={r["dropout"]}  wd={r["weight_decay"]}  mixup={r["mixup_alpha"]}')
    print(f'  macro AUROC        {r["macro_auroc"]:.4f}')
    print(f'  balanced accuracy  {r["balanced_accuracy"]:.4f}')
    print(f'  QWK                {r["quadratic_weighted_kappa"]:.4f}')
    if 'sensitivity' in r:      # 只有二元任務有
        print(f'  sensitivity        {r["sensitivity"]:.4f}')
        print(f'  specificity        {r["specificity"]:.4f}')

## 7. 對照實驗:換 backbone(選跑)

可選:`resnet18` / `resnet34` / `resnet50` / `resnet101` / `efficientnet_b0` / `convnext_tiny`

### ⚠️ 實測結果:更大的 backbone 沒有比較好

| backbone | macro AUROC | QWK |
|---|---|---|
| **resnet18** | **0.9398** | **0.8195** |
| resnet50 | 0.9339 | 0.7828 |

去重後只剩 1074 張訓練影像,**瓶頸是資料量不是模型容量**。往上換 backbone 只會更快過擬合。
所以這節現在是「選跑」—— 除非你想自己確認一次,否則直接跳到第 8 節。

（但這個差距很小,落在雜訊範圍內。要下定論得先做 k-fold cross-validation:
test 只有 231 張、每類約 46 張,單一 split 的 recall 95% CI 大概 ±0.13。)

In [ ]:
%cd /content/test

import json, shutil, pathlib, subprocess

results = {}
try:
    for bb in ['resnet18', 'resnet50', 'convnext_tiny']:
        print(f'\n{"=" * 60}\n  {bb}\n{"=" * 60}')
        subprocess.run(['sed', '-i', f's/^BACKBONE = .*/BACKBONE = "{bb}"/', 'config.py'], check=True)
        subprocess.run(['python', 'train.py'], check=True)   # DATA_DIR 由 os.environ 繼承下去

        rep = json.loads(pathlib.Path('outputs/test_report.json').read_text())
        results[bb] = rep
        shutil.copytree('outputs', f'outputs_{bb}', dirs_exist_ok=True)
finally:
    # 一定要還原,否則 config.py 的 BACKBONE 會停在迴圈最後一個值,
    # 污染之後任何一次 train.py(這正是先前 6b 誤用 convnext_tiny 的原因)
    subprocess.run(['git', 'checkout', '--', 'config.py'], check=True)
    print('\nconfig.py 已還原')

print(f'\n{"backbone":<18}{"macro AUROC":>13}{"bal. acc":>11}{"QWK":>9}')
for bb, r in results.items():
    print(f'{bb:<18}{r["macro_auroc"]:>13.4f}{r["balanced_accuracy"]:>11.4f}'
          f'{r["quadratic_weighted_kappa"]:>9.4f}')

## 7b. 5-fold cross-validation + perceptual dedup

**這次的重點:再去掉一層 leakage。**

md5 去重(exact,1536 張)只抓得到位元組完全相同的圖。但還有約 148 張是
**重新壓縮/縮放過的近重複**(dHash 相同、位元組不同)—— 尤其 F0 特別多:

| dedup | 總數 | F0 | F1 | F2 | F3 | F4 |
|---|---|---|---|---|---|---|
| exact | 1536 | 317 | 296 | 308 | 308 | 307 |
| **perceptual** | **1388** | **199** | 288 | 308 | 308 | 285 |

F0 掉了 118 張近重複 → 這正是它 recall 一直偏高的可疑來源。
`DEDUP=perceptual` 把這層也去掉,是目前能移除的最後一層 exact/near leakage。

跑完拿 perceptual 的 mean ± std 跟你上輪 exact 的 CV(results-3 裡的
`outputs_cv_multiclass` / `outputs_cv_binary`)對照,看數字掉多少。
**預期會掉** —— 掉下來的部分就是近重複原本灌的水。

⚠️ 5 折 × 兩個任務 = 10 次訓練,T4 上約 **30~50 分鐘**,跑之前確認 runtime 不會斷。
⚠️ 仍消不掉 patient-level leakage,數字整體依舊偏樂觀。

In [ ]:
%cd /content/test
!git checkout -- config.py                                   # 清掉第 7 節 sed 過的 BACKBONE
!sed -i 's/^NUM_WORKERS = .*/NUM_WORKERS = 4/' config.py
!grep -E '^(BACKBONE|N_FOLDS|DEDUP|DROPOUT|MIXUP_ALPHA) ' config.py

import json, os, pathlib, subprocess

# 這次只跑 perceptual dedup(1388 張,去掉近重複);exact(1536 張)的 CV 你上輪已經有了。
# 想一次連 exact 一起重跑對照,把 DEDUP_MODES 改成 ['exact', 'perceptual'](時間加倍)。
DEDUP_MODES = ['perceptual']
TASKS = ['multiclass', 'binary_geF2']

summaries = {}
for dedup in DEDUP_MODES:
    for task in TASKS:
        outdir = f'outputs_cv_{dedup}_{task}'
        print(f'\n{"#" * 68}\n#  DEDUP={dedup}  TASK={task}\n{"#" * 68}')
        env = {**os.environ, 'DEDUP': dedup, 'TASK': task, 'CV_RESULTS_DIR': outdir}
        subprocess.run(['python', 'crossval.py'], env=env, check=True)

        s = json.loads((pathlib.Path(outdir) / 'cv_summary.json').read_text())
        assert s['task'] == task, f"summary 寫 task={s['task']},要求的是 {task} → 在跑舊碼"
        summaries[(dedup, task)] = s

print(f'\n{"=" * 68}\n  總結:mean ± std(摺間標準差)\n{"=" * 68}')
for (dedup, task), s in summaries.items():
    print(f'\n[DEDUP={dedup}  {task}]  backbone={s["backbone"]}  '
          f'n_folds={s["n_folds"]}  best_epoch={s["best_epoch_per_fold"]}')
    for k, v in s['scalars'].items():
        print(f'  {k:<26}{v["mean"]:.4f} ± {v["std"]:.4f}')

## 7c. Grad-CAM 混淆稽核

沒有病人 ID → 測試分數是被 leakage 灌水的上限,無法判斷模型有沒有作弊。
而且 F0 有 85% 來自檔名前綴 `a`、`a` 又有 97% 是 F0 —— 模型很可能在認「來源」而非「病理」。
**當你不能相信分數時,唯一能驗證的是「模型在看哪裡」。**

`explain.py` 對每類抽樣做 Grad-CAM,並算「邊緣佔比」(熱圖落在外框 20% 的能量):
- 邊緣佔比高 = 盯著文字標註/探頭刻度/扇形邊緣 → source-confound 的證據
- 邊緣佔比低、熱區在中央肝實質 = 較可信
- **重點:F0 的邊緣佔比是否明顯高於其他類**

先產生一個乾淨的 checkpoint(perceptual multiclass)再稽核。

In [ ]:
%cd /content/test
import os

# 產生乾淨的 checkpoint:perceptual + multiclass,訓練一次存到 checkpoints/best.pt
env = {**os.environ, 'DEDUP': 'perceptual', 'TASK': 'multiclass', 'RESULTS_DIR': 'outputs_audit'}
import subprocess
subprocess.run(['python', 'train.py'], env=env, check=True)

# 稽核:每類抽 6 張做 Grad-CAM,存疊圖 + 邊緣佔比統計
env_e = {**os.environ, 'EXPLAIN_DIR': 'outputs_gradcam', 'PER_CLASS': '6'}
subprocess.run(['python', 'explain.py'], env=env_e, check=True)

## 7d. 校準 + operating point(不用重訓)

吃 7b 存下來的 **out-of-fold 機率**(每張影像由「沒看過它的那一折」預測,是全體資料上
最誠實的一份機率)。**不能**用單一 split 的 test 做校準——樣本太少,且與選模型的資料重疊。

### 為什麼要看校準
臨床上光準確率不夠:模型說「80% 是 F4」時,實際就該有 80% 真的是 F4。
準確率再高、校準爛掉的話,醫師沒辦法拿它的信心值做決策。

本專案有兩股方向相反的力量,不量不知道:
- mixup + label smoothing 通常讓模型**低估**信心
- 但 Grad-CAM 時看到 confidence 幾乎都是 1.00 → 反而像**過度自信**

**ECE**(期望校準誤差)= 依信心分箱,加權平均 |平均信心 − 實際準確率|。0 = 完美。
reliability diagram 上,點落在對角線**下方** = 過度自信。

### 為什麼要看 operating point
二元 ≥F2 現在死用閾值 0.5(sens 0.867 / spec 0.795)。臨床真正的問題是:
**「我要 sensitivity 達到 0.90,閾值該設多少?代價是 specificity 掉到多少?」**
→ 調閾值即可,不用重訓。輸出閾值表讓你挑取捨點。

In [ ]:
%cd /content/test
import subprocess, pathlib

# 吃 7b 產出的 out-of-fold 機率(需先跑過 7b)
for task in ['multiclass', 'binary_geF2']:
    npz = pathlib.Path(f'outputs_cv_perceptual_{task}/oof_predictions.npz')
    assert npz.exists(), f'找不到 {npz} → 請先跑 7b'
    print(f'\n{"#" * 68}\n#  {task}\n{"#" * 68}')
    subprocess.run(['python', 'calibrate.py', str(npz)], check=True)

# reliability.png / roc.png 存在各自的 outputs_cv_* 目錄,會隨第 8 節的 zip 一起下載

## 8. 把結果抓回本機

Colab 執行階段結束後 `/content` 會清空,想留的東西記得下載或存回 Drive。

`checkpoints/best.pt` 有幾十 MB,視需要再抓。

In [ ]:
%cd /content/test
!zip -qr /content/results.zip outputs* checkpoints

from google.colab import files
files.download('/content/results.zip')

# 或存回 Drive(需先在 3b 掛載):
# !cp /content/results.zip '/content/drive/MyDrive/'